# 코드이터 파인튜닝 (Colab)
**순서**: ① 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 → ② 아래 칸을 위에서부터 하나씩 ▶ 실행 → ③ 맨 끝에서 내려받은 파일 2개를 PC에 옮기기.

⚠️ 라이브러리 버전이 바뀌면 중간에 오류가 날 수 있어요. 그때는 오류 글을 그대로 Claude 에게 붙여 주세요.
학습 데이터가 적으면(100~300개) 형식을 익히는 정도이고, 매매 성적이 오른다는 보장은 없어요.

## 1. 설치 (5~10분)

In [ ]:
%%capture
!pip install unsloth

## 2. 파일 올리기
`ce_train_ready.jsonl` 과 `ce_train_eval.jsonl` 두 개를 같이 선택하세요 (ai-trader 폴더의 data 안에 있어요).

In [ ]:
from google.colab import files
up = files.upload()
print(list(up))

## 3. 모델 불러오기 (Qwen3 8B, 4비트)

In [ ]:
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-8B", max_seq_length=1024, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

## 4. 데이터 준비

In [ ]:
import json
from datasets import Dataset
def load(fn):
    return [json.loads(l) for l in open(fn, encoding="utf-8") if l.strip()]
train, ev = load("ce_train_ready.jsonl"), load("ce_train_eval.jsonl")
def to_text(d):
    msgs = [{"role":"user","content":d["instruction"]},{"role":"assistant","content":d["output"]}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, enable_thinking=False)
ds = Dataset.from_list([{"text": to_text(d)} for d in train])
print(len(train), "개로 학습,", len(ev), "개로 평가")
print(ds[0]["text"][:400])

## 5. 학습 (T4 에서 데이터 수백 개면 10~20분)
데이터가 적어서 **2번만** 돌려요. 많이 돌리면 외워 버려요.

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(dataset_text_field="text", per_device_train_batch_size=2, gradient_accumulation_steps=4,
                   num_train_epochs=2, learning_rate=1e-4, warmup_steps=5, logging_steps=5,
                   optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="linear",
                   seed=3407, output_dir="outputs", report_to="none"))
trainer.train()

## 6. 평가용 문제로 확인
학습에 쓰지 않은 문제에 어떻게 답하는지 눈으로 봐요. 정답과 비슷한 형식·내용이면 OK 예요.

In [ ]:
FastLanguageModel.for_inference(model)
for d in ev[:5]:
    ids = tokenizer.apply_chat_template([{"role":"user","content":d["instruction"]}], add_generation_prompt=True,
                                        return_tensors="pt", enable_thinking=False).to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=120, temperature=0.3, do_sample=True)
    print("Q:", d["instruction"][:100]); print("정답:", d["output"][:120]); print("모델:", tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)[:120]); print("-"*40)

## 7. Ollama 용 파일로 저장 (몇 분)
끝나면 `codeeater_ft_gguf` 폴더가 생겨요.

In [ ]:
model.save_pretrained_gguf("codeeater_ft_gguf", tokenizer, quantization_method="q4_k_m")
import os; print(os.listdir("codeeater_ft_gguf"))

## 8. 내려받기
`.gguf` 파일(약 5GB, 몇 분 걸려요)과 `Modelfile` 두 개를 받으세요. 브라우저 다운로드 허용 창이 뜨면 허용하세요.

In [ ]:
import glob
from google.colab import files
for f in glob.glob("codeeater_ft_gguf/*.gguf") + glob.glob("codeeater_ft_gguf/Modelfile"):
    files.download(f)